# 🚬 흡연 분류 AI 해커톤 - V6 최종 (버그 완전 수정)

## ⚠️ V1~V5 치명적 버그 완전 수정!
- **한글 컬럼명 정확히 매핑** (스크린샷 기반)
- 이상치 처리(IQR 클리핑) 복원
- class_weight='balanced' 복원

---

## 📌 STEP 1: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost optuna

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 경로 설정
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 임포트 완료!")

## 📌 STEP 2: 데이터 로드 및 컬럼 확인

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"\n📋 원본 컬럼명:")
print(train.columns.tolist())
print(f"\n📋 처음 2행:")
display(train.head(2))

## 📌 STEP 3: ⭐ 한글 컬럼명 → 영어 매핑 (정확한 매핑!)

In [ ]:
def map_columns_to_english(df):
    """
    한글 컬럼명을 영어로 매핑 (스크린샷 기반 정확한 매핑)
    """
    df = df.copy()
    
    # 정확한 한글 → 영어 매핑 (스크린샷에서 확인된 컬럼명)
    # 키워드 포함 여부로 유연하게 매핑
    col_mapping = {}
    
    for col in df.columns:
        col_lower = col.lower()
        
        # ID
        if 'id' in col_lower:
            col_mapping[col] = 'id'
        # 나이
        elif '나이' in col or 'age' in col_lower:
            col_mapping[col] = 'age'
        # 키
        elif '키' in col or 'height' in col_lower:
            col_mapping[col] = 'height'
        # 몸무게
        elif '몸무게' in col or '체중' in col or 'weight' in col_lower:
            col_mapping[col] = 'weight'
        # BMI
        elif 'bmi' in col_lower or '체질량' in col:
            col_mapping[col] = 'bmi'
        # 시력
        elif '시력' in col or 'eyesight' in col_lower or 'vision' in col_lower:
            col_mapping[col] = 'eyesight'
        # 충치
        elif '충치' in col or '치아' in col or 'cavity' in col_lower or 'dental' in col_lower:
            col_mapping[col] = 'cavity'
        # 공복 혈당
        elif '혈당' in col or '공복' in col or 'blood' in col_lower or 'sugar' in col_lower or 'glucose' in col_lower:
            col_mapping[col] = 'fasting_blood_sugar'
        # 혈압
        elif '혈압' in col or 'pressure' in col_lower:
            col_mapping[col] = 'blood_pressure'
        # 중성 지방
        elif '중성' in col or '지방' in col or 'triglyceride' in col_lower:
            col_mapping[col] = 'triglyceride'
        # 혈청 크레아티닌
        elif '크레' in col or 'creatinine' in col_lower:
            col_mapping[col] = 'serum_creatinine'
        # 콜레스테롤 (총 콜레스테롤)
        elif '콜레스테롤' in col or 'cholesterol' in col_lower:
            col_mapping[col] = 'cholesterol'
        # 고밀도지단백 (HDL)
        elif '고밀도' in col or 'hdl' in col_lower:
            col_mapping[col] = 'hdl'
        # 저밀도지단백 (LDL)
        elif '저밀도' in col or 'ldl' in col_lower:
            col_mapping[col] = 'ldl'
        # 헤모글로빈
        elif '헤모글로빈' in col or 'hemoglobin' in col_lower or 'hb' in col_lower:
            col_mapping[col] = 'hemoglobin'
        # 단백 (요단백)
        elif '단백' in col and '지단백' not in col:
            col_mapping[col] = 'urine_protein'
        # 간 효소율 (GTP, gamma-GTP)
        elif '간' in col or '효소' in col or 'gtp' in col_lower or 'gamma' in col_lower:
            col_mapping[col] = 'gtp'
        # AST
        elif 'ast' in col_lower or 'sgot' in col_lower:
            col_mapping[col] = 'ast'
        # ALT  
        elif 'alt' in col_lower or 'sgpt' in col_lower:
            col_mapping[col] = 'alt'
        # 허리둘레
        elif '허리' in col or 'waist' in col_lower:
            col_mapping[col] = 'waist'
        # label
        elif 'label' in col_lower or '흡연' in col or 'smoking' in col_lower:
            col_mapping[col] = 'label'
        else:
            # 매핑 안 된 컬럼은 원래 이름 유지 (공백, 괄호 제거)
            clean_name = col.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
            col_mapping[col] = clean_name
    
    df = df.rename(columns=col_mapping)
    return df, col_mapping

# 매핑 적용
train_mapped, col_mapping = map_columns_to_english(train)
test_mapped, _ = map_columns_to_english(test)

print("="*60)
print("📊 컬럼명 매핑 결과 (한글 → 영어)")
print("="*60)
for orig, eng in col_mapping.items():
    print(f"  {orig:20} → {eng}")
print(f"\n✅ 매핑 후 컬럼: {train_mapped.columns.tolist()}")

In [ ]:
# ID 처리 및 데이터 분리
train_df = train_mapped.copy()
test_df = test_mapped.copy()

# ID 저장 및 제거
if 'id' in test_df.columns:
    test_id = test_df['id'].copy()
    train_df = train_df.drop('id', axis=1, errors='ignore')
    test_df = test_df.drop('id', axis=1, errors='ignore')
else:
    test_id = pd.Series(range(len(test_df)))

# 타겟 분리
X = train_df.drop('label', axis=1, errors='ignore')
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f"\n📋 매핑된 특성 컬럼: {X.columns.tolist()}")
print(f"\n흡연자 비율: {y.mean()*100:.2f}%")

## 📌 STEP 4: ⭐ 피처 엔지니어링 (이제 제대로 작동!)

In [ ]:
def create_features(df):
    """
    피처 엔지니어링 - 매핑된 영어 컬럼명 사용
    """
    df = df.copy()
    cols = df.columns.tolist()
    
    created = []
    
    # ============================================
    # 1. 콜레스테롤 관련 (흡연자 HDL↓, LDL↑)
    # ============================================
    if 'hdl' in cols and 'ldl' in cols:
        df['hdl_ldl_ratio'] = df['hdl'] / (df['ldl'] + 1)
        df['ldl_hdl_diff'] = df['ldl'] - df['hdl']
        created.extend(['hdl_ldl_ratio', 'ldl_hdl_diff'])
    
    if 'cholesterol' in cols and 'hdl' in cols:
        df['hdl_chol_ratio'] = df['hdl'] / (df['cholesterol'] + 1)
        df['non_hdl_chol'] = df['cholesterol'] - df['hdl']
        df['atherogenic_idx'] = (df['cholesterol'] - df['hdl']) / (df['hdl'] + 1)
        created.extend(['hdl_chol_ratio', 'non_hdl_chol', 'atherogenic_idx'])
    
    if 'triglyceride' in cols and 'hdl' in cols:
        df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
        created.append('tg_hdl_ratio')
    
    # ============================================
    # 2. 간 기능 (흡연자 GTP↑)
    # ============================================
    if 'gtp' in cols:
        df['gtp_log'] = np.log1p(df['gtp'])
        df['gtp_sq'] = df['gtp'] ** 2
        df['gtp_high'] = (df['gtp'] > 50).astype(int)
        created.extend(['gtp_log', 'gtp_sq', 'gtp_high'])
    
    # ============================================
    # 3. 헤모글로빈 (흡연자 높음 - 산소 보상)
    # ============================================
    if 'hemoglobin' in cols:
        df['hemo_sq'] = df['hemoglobin'] ** 2
        df['hemo_log'] = np.log1p(df['hemoglobin'])
        df['hemo_high'] = (df['hemoglobin'] > 15).astype(int)
        df['hemo_vhigh'] = (df['hemoglobin'] > 16).astype(int)
        created.extend(['hemo_sq', 'hemo_log', 'hemo_high', 'hemo_vhigh'])
    
    # 헤모글로빈 × 다른 특성
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['hemo_x_gtp'] = df['hemoglobin'] * df['gtp']
        created.append('hemo_x_gtp')
    
    if 'hemoglobin' in cols and 'triglyceride' in cols:
        df['hemo_x_tg'] = df['hemoglobin'] * df['triglyceride']
        created.append('hemo_x_tg')
    
    # ============================================
    # 4. 혈압 관련
    # ============================================
    if 'blood_pressure' in cols:
        df['bp_high'] = (df['blood_pressure'] > 80).astype(int)
        df['bp_log'] = np.log1p(df['blood_pressure'])
        df['bp_sq'] = df['blood_pressure'] ** 2
        created.extend(['bp_high', 'bp_log', 'bp_sq'])
    
    # ============================================
    # 5. 체형 관련
    # ============================================
    if 'height' in cols and 'weight' in cols:
        height_m = df['height'] / 100
        df['bmi_calc'] = df['weight'] / (height_m ** 2 + 0.01)
        created.append('bmi_calc')
    
    if 'bmi' in cols:
        df['bmi_sq'] = df['bmi'] ** 2
        df['bmi_high'] = (df['bmi'] > 25).astype(int)
        df['bmi_obese'] = (df['bmi'] > 30).astype(int)
        created.extend(['bmi_sq', 'bmi_high', 'bmi_obese'])
    
    # ============================================
    # 6. 시력
    # ============================================
    if 'eyesight' in cols:
        df['eyesight_low'] = (df['eyesight'] < 1.0).astype(int)
        df['eyesight_sq'] = df['eyesight'] ** 2
        created.extend(['eyesight_low', 'eyesight_sq'])
    
    # ============================================
    # 7. 나이 관련
    # ============================================
    if 'age' in cols:
        df['age_sq'] = df['age'] ** 2
        df['age_group'] = pd.cut(df['age'], bins=[0,30,40,50,60,100], labels=[0,1,2,3,4]).astype(float)
        created.extend(['age_sq', 'age_group'])
        
        if 'hemoglobin' in cols:
            df['age_x_hemo'] = df['age'] * df['hemoglobin']
            created.append('age_x_hemo')
        if 'gtp' in cols:
            df['age_x_gtp'] = df['age'] * df['gtp']
            created.append('age_x_gtp')
        if 'triglyceride' in cols:
            df['age_x_tg'] = df['age'] * df['triglyceride']
            created.append('age_x_tg')
    
    # ============================================
    # 8. 혈당 관련
    # ============================================
    if 'fasting_blood_sugar' in cols:
        df['fbs_log'] = np.log1p(df['fasting_blood_sugar'])
        df['fbs_high'] = (df['fasting_blood_sugar'] > 100).astype(int)
        df['fbs_diabetes'] = (df['fasting_blood_sugar'] > 126).astype(int)
        created.extend(['fbs_log', 'fbs_high', 'fbs_diabetes'])
    
    # ============================================
    # 9. 중성지방
    # ============================================
    if 'triglyceride' in cols:
        df['tg_log'] = np.log1p(df['triglyceride'])
        df['tg_sq'] = df['triglyceride'] ** 2
        df['tg_high'] = (df['triglyceride'] > 150).astype(int)
        created.extend(['tg_log', 'tg_sq', 'tg_high'])
    
    # ============================================
    # 10. 크레아티닌
    # ============================================
    if 'serum_creatinine' in cols:
        df['creat_log'] = np.log1p(df['serum_creatinine'])
        df['creat_high'] = (df['serum_creatinine'] > 1.2).astype(int)
        created.extend(['creat_log', 'creat_high'])
    
    # ============================================
    # 11. 종합 건강 점수
    # ============================================
    health_cols = [c for c in cols if c in ['hemoglobin','triglyceride','cholesterol','hdl','ldl','gtp']]
    if len(health_cols) >= 3:
        df['health_mean'] = df[health_cols].mean(axis=1)
        df['health_std'] = df[health_cols].std(axis=1)
        created.extend(['health_mean', 'health_std'])
    
    # ============================================
    # 12. 흡연 위험 점수
    # ============================================
    risk_score = pd.Series(0, index=df.index)
    if 'hemo_high' in df.columns:
        risk_score += df['hemo_high']
    if 'gtp_high' in df.columns:
        risk_score += df['gtp_high']
    if 'tg_high' in df.columns:
        risk_score += df['tg_high']
    if 'bp_high' in df.columns:
        risk_score += df['bp_high']
    df['smoking_risk_score'] = risk_score
    created.append('smoking_risk_score')
    
    # 결측치/무한값 처리
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)
    
    return df, created

# 피처 엔지니어링 적용
print("🔧 피처 엔지니어링 적용 중...")
print(f"원본 특성 수: {X.shape[1]}개")

X_fe, created_features = create_features(X)
X_test_fe, _ = create_features(X_test)

print(f"\n✅ 생성된 파생 피처: {len(created_features)}개")
print(f"   {created_features}")
print(f"\n📊 최종 특성 수: {X_fe.shape[1]}개")

## 📌 STEP 5: ⭐ 이상치 처리 (IQR 클리핑 복원!)

In [ ]:
def clip_outliers(df, multiplier=3.0):
    """
    IQR 기반 이상치 클리핑 (V3에서 복원!)
    """
    df = df.copy()
    clipped_count = 0
    
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - multiplier * IQR
            upper = Q3 + multiplier * IQR
            
            outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            clipped_count += outliers
            
            df[col] = df[col].clip(lower=lower, upper=upper)
    
    print(f"✅ 이상치 클리핑: {clipped_count}개 값 조정")
    return df

X_clipped = clip_outliers(X_fe)
X_test_clipped = clip_outliers(X_test_fe)

## 📌 STEP 6: 모델 튜닝 (⭐ class_weight 복원!)

In [ ]:
print("=" * 50)
print("🔧 하이퍼파라미터 튜닝 (Accuracy 기준)")
print("=" * 50)

# 클래스 불균형 비율
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f"클래스 비율 (비흡연/흡연): {scale_pos_weight:.2f}")

X_np = X_clipped.values
X_test_np = X_test_clipped.values

In [ ]:
# XGBoost
print("\n🔧 XGBoost 튜닝 중...")
xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]  # ⭐ 클래스 가중치!
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='error'),
    xgb_params, n_iter=80, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X_np, y)
best_xgb_params = xgb_search.best_params_
print(f"XGBoost 최고: {xgb_search.best_score_:.5f}")

In [ ]:
# LightGBM
print("\n🔧 LightGBM 튜닝 중...")
lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'class_weight': ['balanced', None]  # ⭐ 클래스 가중치!
}
lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=80, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X_np, y)
best_lgb_params = lgb_search.best_params_
print(f"LightGBM 최고: {lgb_search.best_score_:.5f}")

In [ ]:
# CatBoost
print("\n🔧 CatBoost 튜닝 중...")
cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5],
    'auto_class_weights': ['Balanced', None]  # ⭐ 클래스 가중치!
}
cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X_np, y)
best_cat_params = cat_search.best_params_
print(f"CatBoost 최고: {cat_search.best_score_:.5f}")

In [ ]:
# Random Forest
print("\n🔧 Random Forest 튜닝 중...")
rf_params = {
    'n_estimators': [300, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]  # ⭐ 클래스 가중치!
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, n_iter=40, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
rf_search.fit(X_np, y)
best_rf_params = rf_search.best_params_
print(f"RF 최고: {rf_search.best_score_:.5f}")

In [ ]:
print("\n" + "=" * 50)
print("📊 튜닝 결과 요약")
print("=" * 50)
print(f"XGBoost:  {xgb_search.best_score_:.5f}")
print(f"LightGBM: {lgb_search.best_score_:.5f}")
print(f"CatBoost: {cat_search.best_score_:.5f}")
print(f"RF:       {rf_search.best_score_:.5f}")

## 📌 STEP 7: 10시드 앙상블 + 최적 임계값

In [ ]:
# Train/Val 분할
X_train, X_val, y_train, y_val = train_test_split(
    X_np, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

In [ ]:
print("=" * 50)
print("🎯 10시드 앙상블 학습")
print("=" * 50)

seeds = [42, 123, 456, 789, 1004, 2024, 7777, 8888, 9999, 1234]

val_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for i, seed in enumerate(seeds):
    print(f"Seed {seed} ({i+1}/10)...")
    
    xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='error')
    xgb_m.fit(X_train, y_train)
    val_preds['xgb'].append(xgb_m.predict_proba(X_val)[:, 1])
    
    lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_m.fit(X_train, y_train)
    val_preds['lgb'].append(lgb_m.predict_proba(X_val)[:, 1])
    
    cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_m.fit(X_train, y_train)
    val_preds['cat'].append(cat_m.predict_proba(X_val)[:, 1])
    
    rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_m.fit(X_train, y_train)
    val_preds['rf'].append(rf_m.predict_proba(X_val)[:, 1])

pred_xgb = np.mean(val_preds['xgb'], axis=0)
pred_lgb = np.mean(val_preds['lgb'], axis=0)
pred_cat = np.mean(val_preds['cat'], axis=0)
pred_rf = np.mean(val_preds['rf'], axis=0)

print("\n✅ 10시드 학습 완료!")

In [ ]:
# 최적 가중치 탐색
print("🔍 최적 가중치 탐색 중...")

best_weight_score = 0
best_weights = None

for w1 in np.arange(0.1, 0.6, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        for w3 in np.arange(0.1, 0.6, 0.05):
            w4 = round(1 - w1 - w2 - w3, 2)
            if w4 >= 0.05:
                prob = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf
                pred_label = (prob >= 0.5).astype(int)
                acc = accuracy_score(y_val, pred_label)
                if acc > best_weight_score:
                    best_weight_score = acc
                    best_weights = (w1, w2, w3, w4)

print(f"\n🏆 최적 가중치: XGB={best_weights[0]:.2f}, LGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}, RF={best_weights[3]:.2f}")

In [ ]:
# 최적 임계값 탐색
print("\n" + "=" * 50)
print("🔍 최적 임계값 탐색 (0.01 단위)")
print("=" * 50)

w1, w2, w3, w4 = best_weights
val_prob = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf

best_threshold = 0.5
best_acc = 0
results = []

for thresh in np.arange(0.30, 0.70, 0.01):
    pred_label = (val_prob >= thresh).astype(int)
    acc = accuracy_score(y_val, pred_label)
    f1 = f1_score(y_val, pred_label)
    results.append({'threshold': thresh, 'accuracy': acc, 'f1': f1})
    
    if acc > best_acc:
        best_acc = acc
        best_threshold = thresh

results_df = pd.DataFrame(results)
print("\n상위 10개 임계값:")
print(results_df.nlargest(10, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_threshold:.2f}")
print(f"   Validation Accuracy: {best_acc:.5f}")

## 📌 STEP 8: 최종 예측

In [ ]:
print("=" * 50)
print("📝 최종 모델 학습 (전체 데이터)")
print("=" * 50)

final_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for i, seed in enumerate(seeds):
    print(f"Seed {seed} ({i+1}/10) 최종 학습 중...")
    
    xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='error')
    xgb_m.fit(X_np, y)
    final_preds['xgb'].append(xgb_m.predict_proba(X_test_np)[:, 1])
    
    lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_m.fit(X_np, y)
    final_preds['lgb'].append(lgb_m.predict_proba(X_test_np)[:, 1])
    
    cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_m.fit(X_np, y)
    final_preds['cat'].append(cat_m.predict_proba(X_test_np)[:, 1])
    
    rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_m.fit(X_np, y)
    final_preds['rf'].append(rf_m.predict_proba(X_test_np)[:, 1])

print("\n✅ 최종 학습 완료!")

In [ ]:
# 앙상블 + 임계값 적용
test_xgb = np.mean(final_preds['xgb'], axis=0)
test_lgb = np.mean(final_preds['lgb'], axis=0)
test_cat = np.mean(final_preds['cat'], axis=0)
test_rf = np.mean(final_preds['rf'], axis=0)

test_prob = w1*test_xgb + w2*test_lgb + w3*test_cat + w4*test_rf
final_prediction = (test_prob >= best_threshold).astype(int)

print(f"🎯 사용된 임계값: {best_threshold:.2f}")
print(f"\n예측 결과 분포:")
print(f"   0 (비흡연): {(final_prediction == 0).sum()}명 ({(final_prediction == 0).mean()*100:.1f}%)")
print(f"   1 (흡연):   {(final_prediction == 1).sum()}명 ({(final_prediction == 1).mean()*100:.1f}%)")

## 📌 STEP 9: 제출 파일 생성

In [ ]:
submission_df = submission.copy()
submission_df['label'] = final_prediction
submission_df['label'] = submission_df['label'].astype(int)

print("📋 제출 파일 미리보기:")
display(submission_df.head(10))

# 저장
output_path = result_path + 'submission_v6_final.csv'
submission_df.to_csv(output_path, index=False)
print(f"\n✅ 저장 완료: {output_path}")

In [ ]:
# 검증
print("🔍 제출 파일 검증:")
print(f"   행 개수: {len(submission_df)}")
print(f"   label 타입: {submission_df['label'].dtype}")
print(f"   label 값: {sorted(submission_df['label'].unique())}")

if submission_df['label'].dtype in ['int64','int32'] and set(submission_df['label'].unique()).issubset({0,1}):
    print("\n✅ 검증 통과!")
else:
    print("\n⚠️ 검증 실패!")

## 📌 STEP 10: 다운로드

In [ ]:
from google.colab import files
files.download(output_path)

print("\n" + "=" * 60)
print("🎉 V6 최종 완료!")
print("=" * 60)
print(f"\n📊 V6 핵심 수정사항:")
print(f"   ✅ 한글 컬럼명 → 영어 정확히 매핑")
print(f"   ✅ 피처 엔지니어링 {len(created_features)}개 생성!")
print(f"   ✅ 이상치 처리(IQR) 복원")
print(f"   ✅ class_weight='balanced' 복원")
print(f"\n📊 설정값:")
print(f"   - 임계값: {best_threshold:.2f}")
print(f"   - 가중치: XGB={w1:.2f}, LGB={w2:.2f}, CAT={w3:.2f}, RF={w4:.2f}")
print(f"   - 피처 수: {X_fe.shape[1]}개")
print(f"   - Val Accuracy: {best_acc:.5f}")
print(f"\n🚀 제출하세요!")